# MIG Cement Demand Forecasting & Inventory Optimisation
**Oluwadamilare Akinwumi Tijani**

I use this case study to compare statistical and machine-learning forecasts for daily cement consumption and translate them into an inventory planning scenario. My original group compared AR, SARIMAX and LightGBM; Random Forest was explored in a drop-in session. This edition standardises the evaluation and adds a seasonal-naive baseline. I retain the original notebook filename; previous versions remain in Git history.

The eight-week horizon, 15% MAPE target, 98% pour-readiness target, 20% utilisation improvement and 30% write-off reduction come from the case-study brief. These are objectives, not achieved outcomes.

## 1. Load and validate the source
I connect to the authorised SQLite file in read-only mode. The validation checks daily completeness, uniqueness, foreign keys, physical quantities and consistent site capacity. The database is not included in GitHub. Change the path below for your local copy. I avoid displaying individual records in the published notebook.

In [ ]:
from pathlib import Path
import pandas as pd
from forecast import load_data, features, benchmark, metrics, ORIGINS, HORIZON
from inventory import simulate, summarise
database = Path('MIG_Cement_Records.db')
d = load_data(database)
print({'records': len(d), 'sites': d.site_id.nunique(), 'days': d.date.nunique()})

## 2. Explore operational conditions
I evaluate inventory balance, capacity violations and days with planned work but no recorded consumption. Zero consumption may represent genuine inactivity or a supply constraint; I do not treat it as missing data. Observed consumption is not necessarily unconstrained demand.

In [ ]:
balance = (d.opening_inventory_tonnes + d.deliveries_tonnes - d.consumed_tonnes - d.closing_inventory_tonnes).round(2)
pd.Series({'zero_consumption_days': int(d.consumed_tonnes.eq(0).sum()),
           'planned_but_no_consumption': int(((d.planned_pour_tonnes > 0) & d.consumed_tonnes.eq(0)).sum()),
           'closing_above_capacity': int((d.closing_inventory_tonnes > d.site_capacity).sum()),
           'balance_discrepancies': int(balance.ne(0).sum())})

In [ ]:
monthly = d.set_index('date').resample('MS')[['consumed_tonnes','planned_pour_tonnes']].sum()
monthly.plot(figsize=(11,4), title='Network-wide monthly consumption and planned pours', ylabel='Tonnes');

## 3. Design forecast-time features
I shift historical consumption, planned pours, weather and inventory before calculating features. Future observed weather and future actual inventory are excluded. Where vintage planning snapshots are unavailable, I use historical planning signals. The pooled tree models learn across sites and predict all 56 horizon days directly; statistical models forecast each site separately. Calendar variables are known ahead of time.

In [ ]:
x = features(d)
print('Forecast feature count:', len(x.columns)-2)
print('Forecast horizon:', HORIZON)

## 4. Compare competing models
I train only on observations before each forecast origin. The first two disjoint 56-day windows are used for model selection, and the final window is used for holdout assessment. Seasonal-naive repeats the latest week, AR(7) recursively forecasts from its own predictions, SARIMAX includes known calendar regressors, and the tree models use direct horizon-conditioned forecasts. Failed statistical fits are recorded rather than silently replaced. MAE is the primary selection measure; WAPE, RMSE, bias and nonzero-day MAPE provide additional context.

In [ ]:
predictions, failures = benchmark(d)  # May take several minutes.
pd.DataFrame(metrics(predictions)).round(3)

In [ ]:
validation = predictions[predictions.date < ORIGINS[-1]]
holdout = predictions[predictions.date == ORIGINS[-1]]
expected = 2 * HORIZON * d.site_id.nunique()
eligible = [m for m in metrics(validation) if m['observations'] == expected]
winner = eligible[0]['model']
assert len(holdout[holdout.model == winner]) == HORIZON*d.site_id.nunique()
print('Selected model:', winner, '| Failed fits:', len(failures))
pd.DataFrame(metrics(holdout)).round(3)

## 5. Translate forecasts into inventory decisions
I replay the final holdout using identical initial stocks and observed consumption. I assume a two-day delivery lead and select buffer days on validation. Opening stock above reported capacity is explicitly segregated under a hypothetical capacity-feasible scenario, not written off. I track outstanding orders and overflow. Scheduled-pour availability is a proxy for readiness; it does not prove actual pours were completed. Write-off data are unavailable.

In [ ]:
candidates = []
for buffer in [2,4,6,8,10,14]:
    result = summarise(simulate(d, validation, winner, buffer_days=buffer))
    result['buffer_days'] = buffer
    candidates.append(result)
feasible = [s for s in candidates if s['pour_readiness_pct'] >= 98]
chosen = min(feasible,key=lambda s:s['mean_closing_tonnes']) if feasible else max(candidates,key=lambda s:s['pour_readiness_pct'])
buffer = chosen['buffer_days']
replays = [simulate(d, holdout, m, buffer_days=buffer) for m in dict.fromkeys(['Seasonal naive',winner])]
pd.DataFrame([summarise(z) for z in replays]).drop(columns=['opening_excess_tonnes']).round(3)

## 6. Decision-support application
I keep site-level forecast and inventory output files local. The exported files feed the standalone Plotly Dash application with an executive view, forecasting view and inventory planning view. Follow the README to start the local dashboard. The simulation is a historical research demonstration, not a live procurement system.

## 7. Conclusion
I assess the actual outcomes against the original case-study targets in `docs/results.md`. The final model decision is grounded in the executed comparison rather than the optimistic results of the original leakage-prone evaluation. Before committing, clear all notebook outputs to prevent accidental publication of restricted records.